# CPDS-AI: YOLOv8 Adult vs Child Classification

This notebook runs on Kaggle or Google Colab. It downloads a Roboflow dataset, fine-tunes **YOLOv8n**, validates artifacts, and exports ONNX. It never hard-codes Ultralytics output paths because those paths may change between releases.

In [ ]:
!pip install -q ultralytics roboflow onnx
import ultralytics
ultralytics.checks()

## 1. Download the Dataset from Roboflow

In Kaggle, create a `ROBOFLOW_API_KEY` secret in **Add-ons → Secrets** and grant this notebook access. Never write an API key directly in the notebook or GitHub repository.

In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError("Create Kaggle Secret ROBOFLOW_API_KEY and grant this notebook access.")

rf = Roboflow(api_key=api_key)
project = rf.workspace("timii-owolabi-pwfjm").project("child-adult-detection-bgjzk")
version = project.version(10)
dataset = version.download("yolov8")
dataset_path = Path(dataset.location)
data_yaml = dataset_path / "data.yaml"
if not data_yaml.is_file():
    raise FileNotFoundError(f"Roboflow export is missing data.yaml: {data_yaml}")

print("Dataset path:", dataset_path)

## 2. Train YOLOv8n

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
# Use device='cpu' when no Kaggle GPU accelerator is enabled.
results = model.train(
    data=str(data_yaml), epochs=20, imgsz=640, device=0,
    project="/kaggle/working/cpds_runs", name="adult_child", exist_ok=True,
    patience=10, seed=42,
)

# Never construct runs/... paths manually.
run_dir = Path(results.save_dir)
best_weights = run_dir / "weights" / "best.pt"
if not best_weights.is_file():
    raise FileNotFoundError(f"Training did not create best.pt: {best_weights}")
print(f"Best weights: {best_weights}")

## 3. Export and Smoke-Test ONNX

Final artifacts are copied to `/kaggle/working/artifacts/` for download from Kaggle Output.

In [ ]:
import json
import shutil
import onnx

# Recovery path: export an already-finished Kaggle run without retraining.
if "best_weights" not in globals():
    try:
        best_weights = Path(results.save_dir) / "weights" / "best.pt"
    except (NameError, AttributeError):
        candidates = sorted(Path("/kaggle/working").rglob("best.pt"), key=lambda path: path.stat().st_mtime)
        if not candidates:
            raise FileNotFoundError("No best.pt found. Run the training cell first.")
        best_weights = candidates[-1]
if not Path(best_weights).is_file():
    raise FileNotFoundError(f"Missing best weights: {best_weights}")

best_model = YOLO(str(best_weights))
# A fixed input shape is faster and more predictable on edge ONNX Runtime.
export_path = Path(best_model.export(format="onnx", imgsz=640, opset=12, dynamic=False, simplify=True))
onnx.checker.check_model(str(export_path))

artifacts_dir = Path("/kaggle/working/artifacts")
artifacts_dir.mkdir(exist_ok=True)
onnx_path = artifacts_dir / "yolov8n-adult-child.onnx"
shutil.copy2(export_path, onnx_path)
metadata = {"class_names": best_model.names, "imgsz": 640, "source_weights": str(best_weights)}
(artifacts_dir / "vision_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

sample_images = list((dataset_path / "valid" / "images").glob("*"))
if sample_images:
    smoke_result = YOLO(str(onnx_path), task="detect")(str(sample_images[0]), verbose=False)[0]
    print(f"ONNX smoke test passed: {len(smoke_result.boxes)} detections on {sample_images[0].name}")
print(f"Download these Kaggle outputs: {onnx_path} and {artifacts_dir / 'vision_metadata.json'}")